# 01 — Descripteurs

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- expliquer le protocole descripteur (`__get__`, `__set__`, `__delete__`) ;
- distinguer data descriptors et non-data descriptors, et leur priorité dans la résolution d'attribut ;
- utiliser `__set_name__` pour connaître le nom de l'attribut dans la classe hôte ;
- écrire des descripteurs de validation et de cache (`cached_property` like) ;
- savoir **quand** un descripteur est le bon outil — et quand `@property` suffit.


## Prérequis — ce que vous connaissez déjà

Ce notebook s'adresse à un développeur Python **confirmé**. Vous maîtrisez déjà :

- le modèle objet complet (héritage, MRO, `super`, méthodes spéciales) ;
- les type hints modernes (`int | None`, génériques, `Protocol`, `TypeVar`) ;
- les dataclasses (`@dataclass`, `field`, `frozen=True`, `slots=True`) ;
- les décorateurs de fonction et de classe, et les gestionnaires de contexte ;
- les tests avec `pytest` (fixtures, paramétrage, monkeypatch) ;
- le packaging avec `pyproject.toml` et `uv` ;
- le logging (module `logging`, handlers, formatters) ;
- les bases de SQL et `sqlite3`, les expressions régulières.

Notions que nous allons **introduire ou approfondir** ici :

- le **protocole descripteur** complet et sa place dans la recherche d'attribut ;
- la différence data / non-data et ses conséquences ;
- les cas d'usage concrets : validation, cache, champs ORM.


## Plan

1. Du `@property` au descripteur générique
2. Le protocole : `__get__`, `__set__`, `__delete__`
3. `__set_name__` et la découverte du nom de l'attribut
4. Data descriptors vs non-data descriptors
5. Cas d'usage 1 : validation typée et paramétrée
6. Cas d'usage 2 : cache mémoïsé (`cached_property`)
7. Résolution d'attribut : ordre précis
8. Pièges courants
9. Synthèse
10. Exercices


---

## 1. Du `@property` au descripteur générique

Vous connaissez `@property` : il permet d'exposer une méthode comme un attribut. En réalité, `property` **est un descripteur** — une classe qui implémente le protocole descripteur. Un descripteur est donc simplement la généralisation de cette idée : un objet qui contrôle ce qui se passe quand on le lit, l'écrit ou le supprime **depuis une autre classe**.

In [ ]:
class Produit:
    def __init__(self, prix: float) -> None:
        self._prix = prix

    @property
    def prix(self) -> float:
        return self._prix

    @prix.setter
    def prix(self, valeur: float) -> None:
        if valeur < 0:
            raise ValueError("prix négatif interdit")
        self._prix = valeur

p = Produit(10.0)
p.prix = 12.5
print(p.prix)

Le problème avec `@property` apparaît dès qu'on veut **répéter la même logique** sur plusieurs attributs : il faudrait recopier le code de validation partout. Un descripteur permet d'extraire cette logique **une seule fois** et de la réutiliser.

---

## 2. Le protocole descripteur

Un **descripteur** est une classe qui implémente au moins l'une des trois méthodes suivantes :

| Méthode | Signature | Appelée quand… |
|---|---|---|
| `__get__(self, instance, owner)` | lecture | on fait `obj.attr` |
| `__set__(self, instance, value)` | écriture | on fait `obj.attr = v` |
| `__delete__(self, instance)` | suppression | on fait `del obj.attr` |

Le descripteur doit être une **variable de classe** de la classe qui l'utilise (la *owner class*), pas un attribut d'instance.

In [ ]:
class Trace:
    def __get__(self, instance: object, owner: type) -> str:
        print(f"GET sur {type(instance).__name__}")
        return "valeur"

    def __set__(self, instance: object, value: object) -> None:
        print(f"SET {value!r} sur {type(instance).__name__}")

class Client:
    nom = Trace()   # descripteur : variable de classe

c = Client()
c.nom         # déclenche __get__
c.nom = "Alice"  # déclenche __set__

Notez bien :

- `instance` est l'objet sur lequel on accède à l'attribut (`c` dans l'exemple).
- `owner` est la classe qui contient le descripteur (`Client`).
- Quand on accède **depuis la classe** (`Client.nom`), `instance` vaut `None`.

In [ ]:
print(Client.nom)  # accès depuis la classe : instance=None

### `__get__` depuis la classe

Par convention, un descripteur retourne **lui-même** quand `instance is None`, pour permettre l'introspection (`Client.nom` doit renvoyer le descripteur, pas une valeur quelconque).

In [ ]:
class Poli:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        return f"bonjour depuis {type(instance).__name__}"

class Hello:
    msg = Poli()

print(Hello.msg)       # renvoie le descripteur lui-même
print(Hello().msg)     # renvoie la valeur calculée

---

## 3. `__set_name__` — connaître le nom de l'attribut

Un descripteur ne sait pas, a priori, **sous quel nom** il a été assigné dans la classe hôte. Avant Python 3.6, il fallait passer le nom en paramètre à la main. Depuis la **PEP 487**, Python appelle automatiquement `__set_name__(owner, name)` sur le descripteur au moment où la classe hôte est créée.

In [ ]:
class Nomme:
    def __set_name__(self, owner: type, name: str) -> None:
        print(f"installé comme {name!r} dans {owner.__name__}")
        self._nom_public = name
        self._nom_interne = f"_{name}"

class Article:
    titre = Nomme()
    prix = Nomme()

`__set_name__` est la méthode qui permet d'écrire des descripteurs **réutilisables sur plusieurs attributs** : chaque descripteur apprend son propre nom, et peut stocker ses données dans `instance.__dict__` sous un nom privé.

---

## 4. Data descriptors vs non-data descriptors

Un **data descriptor** implémente `__set__` **et/ou** `__delete__` (en plus ou non de `__get__`).
Un **non-data descriptor** n'implémente que `__get__`.

Cette distinction est **cruciale** car elle change l'ordre de résolution d'attribut :

1. Data descriptors de la classe (priorité max)
2. `instance.__dict__`
3. Non-data descriptors de la classe
4. Méthodes et attributs de classe « normaux »

En d'autres termes : un data descriptor **écrase** `instance.__dict__`, un non-data ne l'écrase pas. C'est pourquoi `@property` (data descriptor) empêche toujours l'affectation directe, tandis qu'une fonction (non-data descriptor !) peut être masquée en attribuant un attribut d'instance du même nom.

In [ ]:
class Demo:
    def methode(self):
        return "méthode normale"

d = Demo()
print(d.methode())

# Les fonctions sont des non-data descriptors : on peut masquer
d.methode = lambda: "remplacée par une instance"
print(d.methode())

In [ ]:
class DemoProp:
    @property
    def attribut(self):
        return "depuis la property"

dp = DemoProp()
print(dp.attribut)
# Une property est un data descriptor : impossible de la masquer
try:
    dp.attribut = "autre chose"
except AttributeError as e:
    print("Échec attendu :", e)

**À retenir :** ajouter `__set__` à votre descripteur change fondamentalement son comportement. Si vous ne voulez jamais être écrasé par une valeur d'instance, implémentez au moins `__set__`, même s'il ne fait que lever une erreur.

---

## 5. Cas d'usage 1 — Validation typée et paramétrée

Premier vrai cas d'usage : valider automatiquement les champs d'une classe. Voici un descripteur `Entier` qui accepte un intervalle autorisé.

In [ ]:
class Entier:
    def __init__(self, min: int | None = None, max: int | None = None) -> None:
        self.min = min
        self.max = max

    def __set_name__(self, owner: type, name: str) -> None:
        self._name = f"_{name}"

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, self._name)

    def __set__(self, instance, value) -> None:
        if not isinstance(value, int):
            raise TypeError(f"int attendu, reçu {type(value).__name__}")
        if self.min is not None and value < self.min:
            raise ValueError(f"doit être >= {self.min}")
        if self.max is not None and value > self.max:
            raise ValueError(f"doit être <= {self.max}")
        setattr(instance, self._name, value)

class Personne:
    age = Entier(min=0, max=150)
    score = Entier(min=0)

p = Personne()
p.age = 42
p.score = 100
print(p.age, p.score)

In [ ]:
# Les erreurs sont interceptées à l'affectation
try:
    p.age = -1
except ValueError as e:
    print("refusé :", e)

try:
    p.age = "pas un entier"
except TypeError as e:
    print("refusé :", e)

### Factoriser : un `Validateur` de base

Pour aller plus loin, on peut factoriser la partie `__get__` / `__set__` dans une classe de base abstraite et laisser les sous-classes redéfinir `validate`.

In [ ]:
from abc import ABC, abstractmethod

class Validateur(ABC):
    def __set_name__(self, owner, name: str) -> None:
        self._name = f"_{name}"

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, self._name)

    def __set__(self, instance, value) -> None:
        self.validate(value)
        setattr(instance, self._name, value)

    @abstractmethod
    def validate(self, value) -> None: ...

class NonVide(Validateur):
    def validate(self, value) -> None:
        if not isinstance(value, str) or not value.strip():
            raise ValueError("chaîne non vide attendue")

class User:
    nom = NonVide()

u = User()
u.nom = "Ada"
print(u.nom)
try:
    u.nom = "   "
except ValueError as e:
    print("refusé :", e)

---

## 6. Cas d'usage 2 — Cache mémoïsé (comme `functools.cached_property`)

`functools.cached_property` est un descripteur : il calcule la valeur au premier accès, la stocke dans `instance.__dict__`, et aux accès suivants Python trouve directement la valeur dans l'instance sans rappeler le descripteur. Cela fonctionne parce que `cached_property` est un **non-data** descriptor (pas de `__set__`), donc `instance.__dict__` a priorité.

In [ ]:
from functools import cached_property

class Rapport:
    def __init__(self, donnees: list[int]) -> None:
        self.donnees = donnees

    @cached_property
    def moyenne(self) -> float:
        print("(calcul...)")
        return sum(self.donnees) / len(self.donnees)

r = Rapport(list(range(1_000_000)))
print(r.moyenne)  # calcul
print(r.moyenne)  # cache : pas de (calcul...) affiché

### Réimplémenter `cached_property` en 10 lignes

Pour comprendre en profondeur, voici une version simplifiée :

In [ ]:
class MonCache:
    def __init__(self, fn):
        self.fn = fn
        self.attr_name = None

    def __set_name__(self, owner, name: str) -> None:
        self.attr_name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        valeur = self.fn(instance)
        # On écrit dans __dict__ de l'instance :
        # comme on est non-data descriptor, __dict__ l'emportera
        instance.__dict__[self.attr_name] = valeur
        return valeur

class Exo:
    def __init__(self, n: int) -> None:
        self.n = n

    @MonCache
    def carre(self):
        print("(je calcule)")
        return self.n * self.n

e = Exo(9)
print(e.carre)
print(e.carre)  # pas de (je calcule) : la valeur est dans e.__dict__

---

## 7. Résolution d'attribut — ordre précis

Cette section est **fondamentale** pour déboguer tout code métaprogrammé. Quand on fait `obj.attr`, Python exécute `type(obj).__getattribute__(obj, 'attr')`, qui suit l'ordre suivant :

1. Parcourir le MRO de `type(obj)` à la recherche d'un **data descriptor** nommé `attr`.
2. Si trouvé → appeler son `__get__`. **Stop.**
3. Sinon, regarder dans `obj.__dict__`.
4. Si trouvé → retourner la valeur. **Stop.**
5. Sinon, parcourir à nouveau le MRO à la recherche d'un **non-data descriptor** ou d'une valeur    de classe normale.
6. Si trouvé → `__get__` s'il est descripteur, sinon retourner la valeur.
7. Sinon → `__getattr__` si défini, sinon `AttributeError`.

Retenez la règle simple : **data descriptor > instance > non-data descriptor > classe**.

In [ ]:
class Data:
    def __get__(self, instance, owner): return "depuis data"
    def __set__(self, instance, value): pass

class NonData:
    def __get__(self, instance, owner): return "depuis non-data"

class Boite:
    data = Data()
    nondata = NonData()

b = Boite()
b.__dict__["data"] = "instance"
b.__dict__["nondata"] = "instance"

print(b.data)     # depuis data (data descriptor gagne)
print(b.nondata)  # instance (non-data perd face à __dict__)

---

## 8. Pièges courants

### 8.1. Stocker la valeur dans `self` du descripteur

**Ne faites jamais** `self.valeur = v` dans `__set__` : il n'y a qu'**une seule instance** du descripteur (variable de classe), donc toutes les instances de la owner class partageraient la même valeur. Stockez toujours dans `instance.__dict__` (ou via `setattr`).

In [ ]:
class Casse:
    def __set_name__(self, owner, name): self.name = name
    def __get__(self, instance, owner):
        if instance is None: return self
        return self._partage  # ← partagée entre toutes les instances
    def __set__(self, instance, value):
        self._partage = value  # ← MAUVAIS !

class Bug:
    x = Casse()

a = Bug(); a.x = 1
b = Bug(); b.x = 2
print("a.x =", a.x, "(devrait être 1 !)")

### 8.2. Descripteur oublié en attribut d'instance

Un descripteur stocké dans `instance.__dict__` n'agit **pas** : le protocole ne s'active que quand le descripteur est variable **de classe**.

In [ ]:
class A:
    pass

a = A()
a.prop = property(lambda self: 42)  # stocké dans __dict__ de l'instance
print(a.prop)  # affiche le property, pas 42

---

## 9. Synthèse

| Règle | Conséquence |
|---|---|
| Descripteur = variable **de classe** | Sinon le protocole ne s'applique pas |
| `__set__` présent → **data descriptor** | Priorité max, ne peut pas être masqué |
| `__get__` seul → **non-data descriptor** | Peut être masqué via `__dict__` |
| `__set_name__` donne le nom attribué | Indispensable pour descripteurs réutilisables |
| Stocker dans `instance.__dict__` | Jamais dans `self` du descripteur |
| `@property` / `classmethod` / `staticmethod` | Ce sont des descripteurs |

**Quand utiliser un descripteur plutôt que `@property` ?**

- Quand la même logique est répétée sur plusieurs attributs (validation, cache, champ ORM).
- Quand on écrit un framework qui doit introspecter les champs d'une classe (ex. SQLAlchemy).
- Sinon, `@property` suffit largement.


---

## 10. Exercices

### Exercice 1 — Descripteur `Positif` *(facile)*

Écrire un descripteur `Positif` qui n'accepte que les nombres `int | float` strictement positifs. Toute autre valeur doit lever `ValueError`. Utiliser `__set_name__`.

Tester sur une classe `Geometrie` avec deux attributs `largeur` et `hauteur`.

In [ ]:
# Votre code ici
class Positif:
    ...


<details>
<summary>📖 Voir la correction</summary>

```python
class Positif:
    def __set_name__(self, owner, name: str) -> None:
        self._name = f"_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return getattr(instance, self._name)

    def __set__(self, instance, value) -> None:
        if not isinstance(value, (int, float)) or value <= 0:
            raise ValueError(f"valeur positive attendue, reçu {value!r}")
        setattr(instance, self._name, value)

class Geometrie:
    largeur = Positif()
    hauteur = Positif()

g = Geometrie()
g.largeur = 10
g.hauteur = 5.5
print(g.largeur, g.hauteur)
```

</details>


### Exercice 2 — `TypedField[T]` générique *(moyen)*

Écrire un descripteur `TypedField` générique paramétré par un type `T`. Il doit refuser toute valeur qui n'est pas une instance de `T`. Utiliser la syntaxe générique Python 3.12+ (`class TypedField[T]:`).

Tester avec `TypedField[str]` et `TypedField[int]`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Descripteurs", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
class TypedField[T]:
    def __init__(self, type_: type[T]) -> None:
        self.type_ = type_

    def __set_name__(self, owner, name: str) -> None:
        self._name = f"_{name}"

    def __get__(self, instance, owner=None) -> T:
        if instance is None:
            return self
        return getattr(instance, self._name)

    def __set__(self, instance, value: T) -> None:
        if not isinstance(value, self.type_):
            raise TypeError(f"{self.type_.__name__} attendu, reçu {type(value).__name__}")
        setattr(instance, self._name, value)

class Personne:
    nom = TypedField(str)
    age = TypedField(int)

p = Personne()
p.nom = "Ada"
p.age = 36
print(p.nom, p.age)
```

</details>


### Exercice 3 — Cache à expiration *(difficile)*

Écrire un descripteur `TTLCachedProperty(seconds: float)` qui cache une valeur calculée mais la recalcule automatiquement après `seconds` secondes. Utiliser `time.monotonic()` pour mesurer le temps.

**Attention :** il doit fonctionner correctement avec plusieurs instances de la classe hôte (pas de cache partagé entre instances).

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Descripteurs", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
import time

class TTLCachedProperty:
    def __init__(self, seconds: float) -> None:
        self.seconds = seconds
        self.fn = None

    def __call__(self, fn):
        self.fn = fn
        return self

    def __set_name__(self, owner, name: str) -> None:
        self._slot = f"_ttl_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        data = instance.__dict__.get(self._slot)
        now = time.monotonic()
        if data is None or now - data[1] > self.seconds:
            valeur = self.fn(instance)
            instance.__dict__[self._slot] = (valeur, now)
            return valeur
        return data[0]

class API:
    def __init__(self, nom: str) -> None:
        self.nom = nom
        self.appels = 0

    @TTLCachedProperty(seconds=0.1)
    def status(self) -> str:
        self.appels += 1
        return f"{self.nom}#{self.appels}"

a = API("srv")
print(a.status, a.status)  # deuxième lecture : cache
time.sleep(0.15)
print(a.status)            # cache expiré : recalcul
```

</details>


### Exercice 4 — Mini-ORM en descripteurs *(deep dive)*

Écrire un mini-ORM où :

- Une classe `Model` sert de base.
- Des descripteurs `Column(type_, primary_key=False)` définissent les champs.
- La classe `Model` a une méthode de classe `fields()` qui renvoie la liste des noms de colonnes   (découverts par introspection de `__dict__`).
- Une méthode `to_dict()` qui renvoie les valeurs d'instance.

**Exigence :** pas de métaclasse. Utiliser uniquement descripteurs + `__init_subclass__` (vu au notebook suivant — vous pouvez aussi le faire sans).

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Descripteurs", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
class Column:
    def __init__(self, type_: type, primary_key: bool = False) -> None:
        self.type_ = type_
        self.primary_key = primary_key

    def __set_name__(self, owner, name: str) -> None:
        self._name = name
        self._slot = f"_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return getattr(instance, self._slot, None)

    def __set__(self, instance, value) -> None:
        if value is not None and not isinstance(value, self.type_):
            raise TypeError(f"{self._name}: {self.type_.__name__} attendu")
        setattr(instance, self._slot, value)

class Model:
    @classmethod
    def fields(cls) -> list[str]:
        return [k for k, v in vars(cls).items() if isinstance(v, Column)]

    def to_dict(self) -> dict:
        return {k: getattr(self, k) for k in self.fields()}

class Utilisateur(Model):
    id = Column(int, primary_key=True)
    nom = Column(str)
    email = Column(str)

u = Utilisateur()
u.id = 1
u.nom = "Ada"
u.email = "ada@example.org"
print(Utilisateur.fields())
print(u.to_dict())
```

</details>


---

## Ressources

- [docs Python — Descriptor HowTo](https://docs.python.org/3/howto/descriptor.html)
- [PEP 487 — `__set_name__`](https://peps.python.org/pep-0487/)
- *Fluent Python* (L. Ramalho), chapitre 23 — "Attribute descriptors"
- Code source de `functools.cached_property` dans CPython
